# IE500618 – Assignment 1: Housing Price Prediction

**Group members:** Fillip Grüner, Kacper Fendrykowski and Julianne Mandal

The goal of this notebook is to predict house sale prices (`SalePrice`) in the supplied Ames Housing dataset (`AmesHousing.csv`). We follow the workflow from the assignment:

**Explore and understand the data → Split the data → Preprocess → Select/engineer features → Train models → Evaluate and interpret**

Each step contains the code and outputs, together with short explanations of what we found and why we made our decisions. When column names or category codes were unclear, we used the *Ames Housing – Understanding the Dataset* guide.

In [ ]:
#all imports used in this notebook
import sys
from pathlib import Path
import numpy as np
import seaborn as sns

src_path = Path.cwd().parent / "src"
sys.path.append(str(src_path.resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.model_selection import train_test_split        # from scikit-learn lib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from config import PATH_AMESHOUSING

pd.set_option("display.max_rows", 100)

## Step 1 – Explore and understand the data

Before building any models we explore the dataset to understand the variables, the target variable `SalePrice`, missing values, unusual values and which relationships may be useful for prediction. These findings are used to support the decisions in the later steps.

### 1a) Dataset structure

First we load the data and look at the columns, their data types and how many non-missing values each column has.

In [ ]:
df = pd.read_csv(PATH_AMESHOUSING)

# all columns with their data types and number of non-null values
df.info()

**Findings:**

- The dataset contains **2930 houses (rows)** and **82 variables (columns)**, including the target variable `SalePrice`.
- There are 43 text (categorical) columns, 28 integer columns and 11 float columns. The variables describe the size, quality, age, location and condition of the houses, as well as details about the sale.
- `Order` and `PID` are only identifiers (row number and parcel ID). They do not describe the house, so they should not be used as features.
- `MS SubClass` is stored as a number, but according to the dataset guide it is a code for the type of dwelling. It should therefore be treated as a categorical variable, not as a numerical one.
- Several columns have fewer than 2930 non-null values, which means they contain missing values. These are looked at in 1c.
- The categorical columns must be encoded before they can be used in the models (Step 3).

### 1b) Distribution of `SalePrice`

`SalePrice` is the target variable, so we look at its distribution with a histogram and a box plot. This tells us whether the prices are symmetric, skewed or contain extreme values.

In [ ]:
sale_price = df["SalePrice"]
# histogram and box plot of the target variable

plt.figure(figsize=(14, 10))
sns.histplot(sale_price, bins=100, edgecolor="black", alpha=1)
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.title("Distribution of Sale Prices")
plt.xlabel("Sale Price (USD)")
plt.ylabel("Amount")
plt.show()

median = sale_price.median()

plt.figure(figsize=(12, 4))
sns.boxplot(x=sale_price, width=0.15, saturation=1, flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25},
    boxprops={"edgecolor": "black"})
plt.gca().xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.text(
    median, 0.2,
    f"Median: ${median:,.0f}",
    ha="center"
)
plt.title("Sale Price", fontweight="bold")
plt.xlabel("Sale Price (USD)")
plt.show()

In [ ]:
# summary statistics for SalePrice
sale_summary = sale_price.describe().to_frame().T
display(sale_summary.style.format("{:,.0f}"))

print(f"Skewness of SalePrice: {sale_price.skew():.2f}")

**Findings:**

The histogram shows that most sale prices are in the lower and middle part of the range, while only a few houses have very high prices. The distribution is therefore **right-skewed**. The positive skewness value in the table above confirms this.

The mean (about $180,800) is higher than the median ($160,000). This confirms that the expensive houses pull the average up.

The box plot shows many unusually high sale prices. The maximum price of $755,000 is far above the third quartile of $213,500. These houses may have a strong effect on some models, especially Linear Regression. We do **not** remove them automatically because they may be valid, expensive houses. Instead we keep them in mind for Step 3, where a log transformation of `SalePrice` could be one way to reduce the skewness.

### 1c) Missing values

Next we count the missing values in each column. Only columns with at least one missing value are shown. A logarithmic y-axis is used because the number of missing values varies from a handful to almost all rows.

In [ ]:
NanValues = df.isnull().sum()
NanValues = NanValues[NanValues > 0].sort_values(ascending=False)

plt.figure(figsize=(12, 6))
bars = plt.bar(NanValues.index, NanValues.values)

plt.yscale('log')
plt.xticks(rotation=45)
plt.ylabel("Number of missing values (log scale)")
plt.xlabel("Column")
plt.title("Missing Values per Column")

# Add the actual count on top of each bar
for bar, value in zip(bars, NanValues.values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
              str(value), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

To decide how to handle each column, we list the missing values together with what a missing value means according to the dataset guide.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

# meaning of NA for each column, based on the "Understanding the Dataset" guide
no_garage = "No garage"
no_basement = "No basement"
na_meaning = {
    "Pool QC": "No pool",
    "Misc Feature": "No miscellaneous feature",
    "Alley": "No alley access",
    "Fence": "No fence",
    "Mas Vnr Type": "No masonry veneer (the value 'None' is read as missing by pandas)",
    "Fireplace Qu": "No fireplace",
    "Garage Type": no_garage,
    "Garage Yr Blt": no_garage,
    "Garage Finish": no_garage,
    "Garage Qual": no_garage,
    "Garage Cond": no_garage,
    "Bsmt Qual": no_basement,
    "Bsmt Cond": no_basement,
    "Bsmt Exposure": no_basement,
    "BsmtFin Type 1": no_basement,
    "BsmtFin Type 2": no_basement,
}

missing_table = pd.DataFrame({
    "Missing": missing,
    "Missing (%)": (missing / len(df) * 100).round(1),
    "Data type": df[missing.index].dtypes.astype(str),
    "Meaning of NA": [na_meaning.get(col, "Genuinely missing / unknown") for col in missing.index],
})

display(missing_table)

If a missing value really means that the house does not have the feature, the number of missing values should match the number of houses where the related size or count is 0. We check this for the most important cases.

In [ ]:
# compare "NA in the category column" with "the feature does not exist" in a related numerical column
checks = [
    ("Pool QC", df["Pool Area"] == 0, "Pool Area == 0"),
    ("Fireplace Qu", df["Fireplaces"] == 0, "Fireplaces == 0"),
    ("Garage Type", df["Garage Area"] == 0, "Garage Area == 0"),
    ("Bsmt Qual", df["Total Bsmt SF"] == 0, "Total Bsmt SF == 0"),
    ("Mas Vnr Type", df["Mas Vnr Area"] == 0, "Mas Vnr Area == 0"),
]

check_table = pd.DataFrame([
    {
        "Column with NA": col,
        "Compared with": description,
        "Houses with NA": df[col].isna().sum(),
        "Houses without feature": has_no_feature.sum(),
        "Both": (df[col].isna() & has_no_feature).sum(),
    }
    for col, has_no_feature, description in checks
])
display(check_table)

# the garage and basement columns should be missing for the same houses
garage_cols = ["Garage Type", "Garage Yr Blt", "Garage Finish", "Garage Qual", "Garage Cond"]
bsmt_cols = ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]

print("Houses with a garage type but other garage info missing:")
display(df.loc[df["Garage Type"].notna() & df[garage_cols].isna().any(axis=1),
               garage_cols + ["Garage Cars", "Garage Area"]])

print("Houses with a basement quality but other basement info missing:")
display(df.loc[df["Bsmt Qual"].notna() & df[bsmt_cols].isna().any(axis=1),
               bsmt_cols + ["Total Bsmt SF"]])

**Findings:**

Several columns have missing values. Some columns, like `Pool QC`, `Misc Feature`, `Alley` and `Fence`, are missing for most houses. Others, like `Lot Frontage` and the garage and basement columns, are missing for a smaller share.

The tables show that most missing values do not mean that the information is unknown. They mean that the house **does not have** the feature:

- For the pool and fireplace columns, the houses with NA are the same houses that have no pool or no fireplace.
- The garage and basement columns are missing for (almost) the same houses, and these are the houses without a garage or basement.
- `Mas Vnr Type` looks like it has many missing values, but most of them are houses with the value "None" (no masonry veneer), which pandas reads as missing.

For these columns the missing value carries real information and can be replaced by a category such as "None" (or 0 for areas and counts).

A few values are **genuinely missing**. These are the rows in the last two tables, where a house has a garage or basement but some of the other information is missing, as well as `Lot Frontage`, `Mas Vnr Area`, `Electrical` and the numerical columns with only one or two missing values. These must be filled in (imputed), for example with the median or most common value.

The handling will be decided in Step 3 and learned from the **training data only**, so no information leaks from the test set.

### 1d) Unusual values, patterns and relationships

To find variables that may be useful for predicting house prices, we first calculate the correlation between each numerical variable and `SalePrice`. Correlation only measures *linear* relationships between numerical variables, so categorical variables are not included here and are explored separately with plots.

In [ ]:
# correlation between every numerical variable and SalePrice, sorted from highest to lowest
target_to_compare_to = df.corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)

plt.figure(figsize=(3, 10))
sns.heatmap(target_to_compare_to.to_frame(), cmap="coolwarm", annot=True, fmt=".2f", cbar=True)
plt.title("Correlation with SalePrice", fontweight="bold")
plt.show()

**Findings:**

`Overall Qual` has the strongest correlation with `SalePrice` (0.80), followed by the above-ground living area `Gr Liv Area` (0.71), garage capacity `Garage Cars` (0.65) and `Garage Area` (0.64). Variables such as year built, basement area and number of bathrooms also show moderate positive correlations.

In general, variables describing the **quality, size and age** of the house are the most related to the sale price. These are strong candidates for features in Step 4.

As expected, `Order` and `PID` have almost no correlation with the price, which supports removing them. Some variables are also strongly related to each other, such as `Garage Cars` and `Garage Area`, since both measure the size of the garage. Keeping both may give redundant information.

Correlation shows only a relationship between two variables and does not mean that one causes the other.

Below, some of the strongest relationships are shown in more detail.

In [ ]:
#looking at some of the highest correlated variables to sale price, 
# we can see that the living area, overall quality, year built, and garage area are 
# all highly correlated with sale price. Let's visualize these relationships with scatter plots and box plots.
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="Gr Liv Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[0, 0]
)
axes[0, 0].set_title("Living Area and Sale Price")

sns.boxplot(
    data=df,
    x="Overall Qual",
    y="SalePrice",
    hue="Overall Qual",
    legend=False,
    width=0.5,
    saturation=1,
    palette="viridis",
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"},
    ax=axes[0, 1]
)
axes[0, 1].set_title("Overall Quality and Sale Price")

sns.scatterplot(
    data=df,
    x="Year Built",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 0]
)
axes[1, 0].set_title("Year Built and Sale Price")

sns.scatterplot(
    data=df,
    x="Garage Area",
    y="SalePrice",
    hue="Overall Qual",
    palette="viridis",
    alpha=0.8,
    edgecolor="black",
    linewidth=0.25,
    ax=axes[1, 1]
)
axes[1, 1].set_title("Garage Area and Sale Price")

for ax in axes.flat:
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(
        mtick.StrMethodFormatter('${x:,.0f}')
    )
    ax.title.set_fontweight("bold")
    legend = ax.get_legend()
    if legend:
        legend.set_title("Overall Qual", prop={"size": 10})
        for text in legend.get_texts():
            text.set_fontsize(8)

plt.tight_layout()
plt.show()

In [ ]:
# the very large houses that stand out in the living area plot
large_houses = df.loc[
    df["Gr Liv Area"] > 4000,
    ["Gr Liv Area", "Overall Qual", "SalePrice", "Neighborhood", "Sale Condition", "Yr Sold"]
].sort_values("SalePrice")

display(large_houses)

**Findings:**

The plots show that houses with higher overall quality are usually more expensive, newer and larger, and often have larger garages. However, houses with the same quality rating can still have quite different prices. Overall quality is therefore a useful feature, but it cannot predict the sale price by itself.

The living area plot also shows some **unusual values**: a few very large houses (above 4,000 sq ft) were sold for much lower prices than their size and quality would suggest. The table above lists these houses. The low-priced ones have the sale condition `Partial`, which according to the dataset guide means the house was not completed when it was last assessed. Their prices may therefore not reflect a normal sale. We do not remove them now, but we will decide how to handle them in Step 3 based on the training data.

Next we look at how many houses there are in each quality level.

In [ ]:
# number of houses for each overall quality rating
quality_table = (
    df["Overall Qual"]
    .value_counts()
    .sort_index()
    .rename_axis("Overall Quality")
    .reset_index(name="Number of Houses")
)

display(quality_table)

**Findings:**

Most houses have a quality rating between 4 and 9, while there are very few houses rated 1–3 or 10. Because the models will see few examples of these groups during training, predictions for the lowest and highest quality houses may be less reliable.

#### Categorical quality features

The correlation analysis only includes numerical variables. Many of the categorical variables are quality ratings (Po = poor, Fa = fair, TA = typical/average, Gd = good, Ex = excellent), so we check whether they also separate the sale prices. `Central Air` is included as an example of a simple yes/no feature. The number of houses in each category is shown under each box.

In [ ]:
quality_order = ["Po", "Fa", "TA", "Gd", "Ex"]
cat_features = ["Exter Qual", "Kitchen Qual", "Bsmt Qual", "Central Air"]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)

for ax, col in zip(axes, cat_features):
    if col == "Central Air":
        order = ["N", "Y"]
    else:
        order = [q for q in quality_order if q in df[col].unique()]

    sns.boxplot(
        data=df,
        x=col,
        y="SalePrice",
        order=order,
        hue=col,
        hue_order=order,
        palette="viridis",
        legend=False,
        width=0.5,
        saturation=1,
        flierprops={
            "markersize": 2.5,
            "markerfacecolor": "red",
            "markeredgecolor": "black",
            "alpha": 0.25
        },
        boxprops={"edgecolor": "black"},
        ax=ax
    )

    counts = df[col].value_counts()
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([f"{v}\n(n={counts.get(v, 0)})" for v in order])
    ax.set_title(f"{col} and Sale Price", fontweight="bold")
    ax.set_ylabel("Sale Price (USD)")
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.tight_layout()
plt.show()

**Findings:**

The categorical quality ratings also separate the sale prices clearly. Houses with excellent exterior, kitchen or basement quality are much more expensive than houses rated typical or fair. Houses without central air are generally cheaper. These variables therefore look useful for prediction, even though they are not part of the correlation analysis.

Because the ratings have a natural order (Po < Fa < TA < Gd < Ex), they could be encoded as numbers in Step 3 (ordinal encoding) instead of one column per category. Some categories contain very few houses (see the counts), so the boxes for those categories are less reliable. Houses without a basement have no `Bsmt Qual` value and are not shown in that plot.

### 1e) Sale price by neighbourhood

Location is usually important for house prices, but `Neighborhood` is a categorical variable and is therefore not included in the correlation analysis above. To check whether it could be useful, we compare the sale price distribution in each neighbourhood. The neighbourhoods are sorted by median sale price.

In [ ]:
order_by_price = (
    df.groupby("Neighborhood")["SalePrice"]
      .median()
      .sort_values()
      .index
)

# Sorted by sale price
plt.figure(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Neighborhood",
    y="SalePrice",
    order=order_by_price,
    hue="Neighborhood",
    palette="Spectral",
    legend=False,
    width=0.6,
    saturation=1,
    flierprops={
        "markersize": 2.5,
        "markerfacecolor": "red",
        "markeredgecolor": "black",
        "alpha": 0.25
    },
    boxprops={"edgecolor": "black"}
)

plt.gca().yaxis.set_major_formatter(
    mtick.StrMethodFormatter('${x:,.0f}')
)

plt.xticks(rotation=65, ha="right")
plt.title("Sale Price Distributions by Neighbourhood - Sorted by Price",
          fontweight="bold")
plt.xlabel("Neighbourhood")
plt.ylabel("Sale Price (USD)")
plt.tight_layout()
plt.show()

**Findings:**

The box plots show that sale prices differ clearly between neighbourhoods. Some neighbourhoods have a much higher median sale price and a larger spread than others. This suggests that `Neighborhood` may be a useful feature for predicting house prices, even though it does not appear in the correlation analysis. Since it is categorical, it must be encoded before it can be used (Step 3).

There are also some large outliers within several neighbourhoods. These could be explained by differences in house size, quality or other features, so they are worth looking at more closely.

### Summary of Step 1

- The dataset has 2930 houses and 82 variables, with a mix of numerical and categorical variables. `Order` and `PID` are identifiers and should not be used as features.
- `SalePrice` is right-skewed with some very expensive houses. These are kept, but a log transformation may help, especially for Linear Regression.
- Many missing values mean that the house does not have a feature (e.g. no pool or no garage). Missing values must be handled based on what they represent, using only the training data.
- Quality, size, age and neighbourhood appear to be the most useful information for predicting the price. `Overall Qual`, `Gr Liv Area`, the garage variables, `Year Built` and `Neighborhood` are strong feature candidates.
- There is no clear reason yet to remove any houses, but the very large, low-priced houses and the rare quality levels should be kept in mind in the later steps.

## Step 2 – Create training and test data

In [ ]:
# splitting df into training and testing data
# test_size = 0.2 means 20% is held out for testing
# random_state = 13 means we use the same split every time we run notebook. makes it reproduceable(we can do the same again and again)
train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 13)

print(train_df.shape, test_df.shape)

# from here on, we will use train_df for all exploration/cleaning

## Step 3 – Prepare the data

In this step we prepare the data so it can be used by the three regression models. The most important rule is:

> **Everything that is learned from data (for example a median, the most common category or the scaling) is learned from the training data only, and then applied in the same way to the test data.**

If we used the whole dataset, information from the test houses would leak into the preprocessing, and the test score would no longer be an honest estimate of how the model performs on unseen houses.

Our plan, based on the findings in Step 1:

| Part | What we do | Learned from data? |
|---|---|---|
| 3a | Remove a few unusual houses from the **training set only** | No |
| 3b | Separate features and target, and log-transform `SalePrice` | No |
| 3c | Fill missing values that mean "the house does not have this feature" | No (fixed rules from the dataset guide) |
| 3d | Remove identifier, redundant and almost constant variables | Yes – from the training set |
| 3e | Impute the remaining missing values, encode categorical variables and scale numerical variables | Yes – fitted on the training set |

### 3a) Unusual houses in the training data

In Step 1 we found some very large houses (above 4,000 sq ft) that were sold for much lower prices than their size and quality suggest. These sales have the sale condition `Partial`, which according to the dataset guide means the home was not completed when it was last assessed (often a new-home sale). The price therefore does not reflect a normal sale of a finished house. Such houses would teach the models a wrong relationship between size and price.

We therefore remove these houses **from the training set only**. The test set is left untouched, because it should represent unseen houses as they are. The two very large houses in `NoRidge` with high prices follow the normal pattern and are kept.

In [ ]:
unusual = (train_df["Gr Liv Area"] > 4000) & (train_df["Sale Condition"] == "Partial")
display(train_df.loc[unusual, ["Gr Liv Area", "Overall Qual", "SalePrice", "Neighborhood", "Sale Condition"]])

train_df = train_df.loc[~unusual]
print("Training set after removing unusual houses:", train_df.shape)
print("Test set (unchanged):", test_df.shape)

**Findings:** Two of the unusual houses were in the training set and are removed. A third house of the same kind ended up in the test set. We keep it there, since in a real situation we cannot choose which houses we have to predict. We expect this house to get a large prediction error.

### 3b) Features, target and target transformation

We separate the features (`X`) from the target (`y`). In Step 1 we saw that `SalePrice` is right-skewed with some very expensive houses. We compare the distribution of `SalePrice` in the training set before and after a log transformation, `log(1 + SalePrice)`.

In [ ]:
X_train = train_df.drop(columns="SalePrice")
y_train = train_df["SalePrice"]
X_test = test_df.drop(columns="SalePrice")
y_test = test_df["SalePrice"]

y_train_log = np.log1p(y_train)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(y_train, bins=60, edgecolor="black", alpha=1, ax=axes[0])
axes[0].set_title(f"SalePrice (skewness: {y_train.skew():.2f})", fontweight="bold")
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
axes[0].set_xlabel("Sale Price (USD)")

sns.histplot(y_train_log, bins=60, edgecolor="black", alpha=1, ax=axes[1])
axes[1].set_title(f"log(1 + SalePrice) (skewness: {y_train_log.skew():.2f})", fontweight="bold")
axes[1].set_xlabel("log(1 + Sale Price)")

for ax in axes:
    ax.set_ylabel("Amount")
plt.tight_layout()
plt.show()

**Findings:** `SalePrice` is right-skewed (skewness about 1.8 in the training set). After the log transformation `log(1 + SalePrice)` the distribution is almost symmetric (skewness close to 0).

**Decision:** We train the models to predict `log(1 + SalePrice)` instead of `SalePrice`, for three reasons:

- **Expensive houses dominate otherwise.** The models minimize squared errors, so a large error on one expensive house counts much more than several small errors on cheaper houses. On the log scale, the most expensive houses have less influence.
- **Price effects are relative.** A feature like a better kitchen tends to add a certain *percentage* to the price, not a fixed dollar amount. The log turns percentage changes into equal steps (a 10% increase is the same step for a $100,000 and a $500,000 house), which is easier for Linear Regression to model.
- **No data leakage.** The log is a fixed formula and learns nothing from the data.

Only `y_train` is transformed. The predictions are converted back to dollars with `exp(x) − 1` (`np.expm1`) before evaluation, so the RMSE in Step 6 is in USD and compared with the original `y_test`. The same target transformation is used for all three models, so the comparison is fair.

### 3c) Missing values that mean "the house does not have this feature"

The dataset guide explains that a missing value can mean two different things: either **the feature is not present**, or **the information was not recorded**. We follow the guide for each column:

| Columns | Guide says | What we do here |
|---|---|---|
| `Alley`, `Fence`, `Misc Feature`, `Pool QC`, `Fireplace Qu` | Missing = no alley / fence / feature / pool / fireplace | Fill with the category `"None"` |
| Garage categories (`Garage Type`, `Garage Finish`, `Garage Qual`, `Garage Cond`) | Missing usually means no garage, **unless related garage columns show that a garage exists** | Fill with `"None"` only if the house has no garage area |
| Basement categories (`Bsmt Qual`, `Bsmt Cond`, `Bsmt Exposure`, `BsmtFin Type 1/2`) | Missing usually means no basement, **unless related basement columns show that a basement exists** | Fill with `"None"` only if the house has no basement area |
| `Mas Vnr Type` | Missing and `Mas Vnr Area` = 0 means no veneer. Missing and area > 0 means the type was not recorded | Fill with `"None"` only when the area is 0 |
| Numerical columns (`Lot Frontage`, `Mas Vnr Area`, basement areas and bathrooms, `Garage Cars`, `Garage Area`) | Missing = value not recorded, treat it as an unknown numerical value | Left for the imputer in 3e |
| `Electrical` | Missing = not recorded, treat it as an unknown category | Left for the imputer in 3e |

We also convert `MS SubClass` to text, because the guide describes it as a category (a code for the type of house), not a numerical value.

These rules come from the guide and do not learn anything from the data, so the same function can safely be applied to both the training and the test set.

In [ ]:
garage_cat_cols = ["Garage Type", "Garage Finish", "Garage Qual", "Garage Cond"]
bsmt_cat_cols = ["Bsmt Qual", "Bsmt Cond", "Bsmt Exposure", "BsmtFin Type 1", "BsmtFin Type 2"]


def fill_known_missing(data):
    """Fill missing values that mean 'the feature is not present', following the dataset guide.
    Uses only fixed rules, so nothing is learned from the data.
    Values that the guide describes as 'not recorded' are left for the imputer."""
    data = data.copy()

    # guide: missing = no alley / fence / misc feature / pool / fireplace
    for col in ["Alley", "Fence", "Misc Feature", "Pool QC", "Fireplace Qu"]:
        data[col] = data[col].fillna("No_ne")

    # guide: missing usually means no garage, unless related garage columns show that a garage exists
    no_garage = data["Garage Type"].isna() & ~(data["Garage Area"] > 0)
    data.loc[no_garage, garage_cat_cols] = data.loc[no_garage, garage_cat_cols].fillna("None")

    # guide: missing usually means no basement, unless related basement columns show that a basement exists
    no_bsmt = data["Bsmt Qual"].isna() & ~(data["Total Bsmt SF"] > 0)
    data.loc[no_bsmt, bsmt_cat_cols] = data.loc[no_bsmt, bsmt_cat_cols].fillna("None")

    # guide: Mas Vnr Type missing and Mas Vnr Area = 0 -> no masonry veneer
    # (the csv contains the text 'None' here, which pandas reads as missing)
    no_veneer = data["Mas Vnr Type"].isna() & (data["Mas Vnr Area"] == 0)
    data.loc[no_veneer, "Mas Vnr Type"] = "None"

    # guide: MS SubClass is a category code, not a number
    data["MS SubClass"] = data["MS SubClass"].astype(str)

    return data


X_train = fill_known_missing(X_train)
X_test = fill_known_missing(X_test)

remaining = X_train.isna().sum()
display(remaining[remaining > 0].sort_values(ascending=False).to_frame("Missing in training set"))

**Findings:** After applying the rules from the guide, the missing values that are left in the training set are the ones the guide describes as **not recorded**:

- `Lot Frontage` is the main one. Every house has a lot frontage, but the measurement is missing for many houses.
- `Mas Vnr Area` is not recorded for a few houses, and for those houses `Mas Vnr Type` is unknown as well. A few houses also have a veneer area but no recorded veneer type.
- A few basement values (`Bsmt Exposure`, `BsmtFin Type 2`) are missing for houses that do have a basement.
- `Electrical` and the basement bathroom counts are missing for single houses.

These values are genuinely unknown, so they are imputed in 3e with the median (numerical) or the most common value (categorical), learned from the training set.

`Garage Yr Blt` is also still missing for houses without a garage. There is no sensible year to fill in for a garage that does not exist, and the column is removed in 3d.

### 3d) Removing variables

We remove variables only when there is a clear reason:

- **`Order` and `PID`** are identifiers (row number and parcel ID). They do not describe the house.
- **`Garage Yr Blt`** is missing for all houses without a garage, and there is no sensible value to fill in. It is also strongly correlated with `Year Built`, so it adds little new information. It also contains a typing error (a garage built in the year 2207, see below).
- **Almost constant variables**: if more than 99% of the training houses have the same value, the variable cannot help the models to separate cheap and expensive houses. We compute this on the **training set only** and remove the same columns from the test set.

Variables that are only weakly related to the price are not removed here. That is part of the feature selection in Step 4.

In [ ]:
# evidence for removing Garage Yr Blt
print(f"Correlation between Garage Yr Blt and Year Built (training set): "
      f"{X_train['Garage Yr Blt'].corr(X_train['Year Built']):.2f}")
display(X_train.loc[X_train["Garage Yr Blt"] > 2010, ["Year Built", "Garage Yr Blt", "Yr Sold"]])

# share of the most common value in each column, computed on the training set
most_common_share = X_train.apply(lambda col: col.value_counts(normalize=True, dropna=False).iloc[0])
almost_constant = most_common_share[most_common_share > 0.99].sort_values(ascending=False)
display(almost_constant.to_frame("Share of most common value").style.format("{:.1%}"))

drop_cols = ["Order", "PID", "Garage Yr Blt"] + list(almost_constant.index)

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols)

print("Removed columns:", drop_cols)
print("Training features:", X_train.shape, " Test features:", X_test.shape)

**Findings:** `Garage Yr Blt` is strongly correlated with `Year Built`, and one house has a garage year of 2207, which must be a typing error. Removing the column solves both problems.

The almost constant variables are shown in the table. For example, `Utilities` and `Street` have the same value for nearly every house, and only a handful of houses have a pool (`Pool QC`, `Pool Area`). These columns carry almost no information for predicting the price.

### 3e) encoding and scaling

The remaining variables are divided into three groups, which are prepared in different ways. The dataset guide marks each categorical column as either **Rating** / **Category (ordered)** or plain **Category**, which we use to decide between ordinal and one-hot encoding.

| Group | Examples | Missing values | Transformation |
|---|---|---|---|
| **Numerical** | `Gr Liv Area`, `Lot Frontage`, `Year Built` | median of the training set | standardized (mean 0, standard deviation 1) |
| **Ordinal** (guide: Rating or Category (ordered)) | `Exter Qual`, `Kitchen Qual`, `Bsmt Exposure`, `Garage Finish`, `Functional`, `Paved Drive` | most common value in the training set | converted to numbers that follow the order, e.g. None = 0, Po = 1, …, Ex = 5 |
| **Nominal** (guide: Category) | `Neighborhood`, `MS SubClass`, `House Style` | most common value in the training set | one-hot encoded (one 0/1 column per category) |

Reasons for these choices:

- **Median** is used instead of the mean for numerical values, because many size variables are skewed and the median is less affected by very large houses.
- **Ordinal encoding** keeps the order of the ratings. In Step 1 we saw that the price rises steadily with the quality rating, and one-hot encoding would throw this order away. `Overall Qual` and `Overall Cond` are already stored as numbers 1–10 in the right order, so they are treated as numerical.
- **One-hot encoding** is used for categories without an order, such as neighbourhoods. Giving them numbers like 1, 2, 3 would suggest an order that does not exist. Categories that only appear in the test set are ignored instead of causing an error.
- **Scaling** puts all numerical variables on the same scale. This matters for Linear Regression and does not affect the decision tree based models, so we can use the same prepared features for all three models, as the assignment requires.

All of this is combined in one `ColumnTransformer`, which is **fitted on the training set only** and then used to transform both sets.

In [ ]:
# order of the categories, from worst to best, taken from the dataset guide
quality_scale = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
ordinal_categories = {
    "Exter Qual": quality_scale,
    "Exter Cond": quality_scale,
    "Bsmt Qual": quality_scale,
    "Bsmt Cond": quality_scale,
    "Heating QC": quality_scale,
    "Kitchen Qual": quality_scale,
    "Fireplace Qu": quality_scale,
    "Garage Qual": quality_scale,
    "Garage Cond": quality_scale,
    "Pool QC": quality_scale,
    "Bsmt Exposure": ["None", "No", "Mn", "Av", "Gd"],
    "BsmtFin Type 1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "BsmtFin Type 2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    "Garage Finish": ["None", "Unf", "RFn", "Fin"],
    "Lot Shape": ["IR3", "IR2", "IR1", "Reg"],
    "Land Slope": ["Sev", "Mod", "Gtl"],
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    "Paved Drive": ["N", "P", "Y"],
    "Utilities": ["NoSeWa", "NoSewr", "AllPub"],
}
# only keep the ordinal columns that were not removed in 3d
ordinal_categories = {col: cats for col, cats in ordinal_categories.items() if col in X_train.columns}

ordinal_cols = list(ordinal_categories)
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
nominal_cols = [col for col in X_train.columns if col not in numerical_cols + ordinal_cols]

print(f"{len(numerical_cols)} numerical, {len(ordinal_cols)} ordinal and {len(nominal_cols)} nominal columns")

numerical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

ordinal_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OrdinalEncoder(categories=list(ordinal_categories.values()),
                              handle_unknown="use_encoded_value", unknown_value=-1)),
])

nominal_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("ord", ordinal_pipeline, ordinal_cols),
    ("nom", nominal_pipeline, nominal_cols),
], verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

# learn the medians, most common values, categories and scaling from the TRAINING set only
preprocessor.fit(X_train)

# apply the same learned preprocessing to both sets
X_train_prepared = preprocessor.transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Prepared training data:", X_train_prepared.shape)
print("Prepared test data:    ", X_test_prepared.shape)
print("Missing values left - training:", X_train_prepared.isna().sum().sum(),
      " test:", X_test_prepared.isna().sum().sum())

display(X_train_prepared.head())

**Findings:** After preprocessing, the training and test sets have exactly the same columns and no missing values. The number of columns has increased, because each nominal variable has been split into one 0/1 column per category.

The medians, the most common values, the list of categories and the scaling were all learned from the training set. The test set was only transformed, never used for fitting. The prepared data are now ready for feature selection in Step 4 and model training in Step 5. In Step 5 the `preprocessor` can be put in a `Pipeline` together with each model, so that the same preprocessing is always fitted on the training data only.

### Summary of Step 3

- **Unusual houses:** two very large, partially completed houses were removed from the training set only. The test set is untouched.
- **Target:** `SalePrice` is log-transformed to reduce the skewness. Predictions are converted back to USD before evaluation.
- **Missing values:** following the dataset guide, missing values that mean "no pool / garage / basement …" get the category `"None"`. Values the guide describes as "not recorded" (for example `Lot Frontage`, `Mas Vnr Area` and `Electrical`) are imputed with the median or most common value of the training set.
- **Removed variables:** identifiers (`Order`, `PID`), `Garage Yr Blt` (redundant, missing for houses without a garage and contains an error) and almost constant variables.
- **Encoding and scaling:** ratings and ordered categories (as marked in the guide) are ordinal encoded, other categories are one-hot encoded, and numerical variables are standardized.
- **No leakage:** everything that is learned from data is learned from the training set and then applied unchanged to the test set.